In [41]:
import os, asyncio, httpx, pyperclip
import pandas as pd
from binance import AsyncClient, BinanceSocketManager
from dotenv import load_dotenv

from exchange.Trades import Order


load_dotenv('../../.env')
BINANCE_KEY = os.getenv('BINANCE_KEY')
BINANCE_SECRET = os.getenv('BINANCE_SECRET')

client = await AsyncClient.create(BINANCE_KEY, BINANCE_SECRET)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x72d93dd390a0>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x72d940165010>, 6598.411599097)])']
connector: <aiohttp.connector.TCPConnector object at 0x72d941940290>


## Orders

In [34]:
async def clean_orders(df_: pd.DataFrame):
    df_['time'] = pd.to_datetime(df_['time'], unit='ms')
    df_['updateTime'] = pd.to_datetime(df_['updateTime'], unit='ms')
    df_['workingTime'] = pd.to_datetime(df_['workingTime'], unit='ms')
    df_ = df_.drop(columns=['workingTime', 'selfTradePreventionMode', 'isWorking', 'orderListId'])
    df_ = df_.rename(columns={
        'clientOrderId': 'client_orderid',
        'origQty': 'amount',
        'executedQty': 'executed_amount',
        'cummulativeQuoteQty': 'cum_quote_amount',
        'timeInForce': 'time_in_force',
        'stopPrice': 'stop_price',
        'icebergQty': 'iceberg_amount',
        'origQuoteOrderQty': 'quote_amount',
        'time': 'created_at',
        'updateTime': 'updated_at',
    })
    # df_[df_['price'] == '0.00000000']
    return df_

def df_to_sql_insert(df, table_name="orders"):
    columns = ','.join(df.columns)
    insert_template = f"INSERT INTO {table_name} ({columns}) VALUES "
    values = [
        f"({','.join([repr(val) for val in row])})"
        for row in df.itertuples(index=False)
    ]
    return insert_template + ',\n'.join(values) + ";"

In [35]:
basedf = pd.DataFrame(await client.get_all_orders(symbol='BANANAUSDT')).set_index('orderId')
ordersdf = await clean_orders(basedf.copy())
# ordersdf.loc[386446134]
ordersdf

,symbol,client_orderid,price,amount,executed_amount,cum_quote_amount,status,time_in_force,type,side,stop_price,iceberg_amount,created_at,updated_at,quote_amount
orderId,,,,,,,,,,,,,,,
376714213,BANANAUSDT,web_5e40a14cfb674c2ab1ee8acdcdec25af,22.90000000,87.33600000,0.00000000,0.00000000,CANCELED,GTC,STOP_LOSS_LIMIT,BUY,22.86000000,0.00000000,2025-03-25 22:39:45.828,2025-03-25 22:41:31.773,0.00000000
376715538,BANANAUSDT,web_0667b2c4968b4640a051d04714b6b521,22.90000000,87.37900000,0.00000000,0.00000000,CANCELED,GTC,STOP_LOSS_LIMIT,BUY,22.86000000,0.00000000,2025-03-25 22:43:03.776,2025-03-26 00:42:57.982,0.00000000
383577752,BANANAUSDT,web_7e046e2abb464590aa91f928f16e87e4,19.44000000,101.15600000,0.00000000,0.00000000,CANCELED,GTC,STOP_LOSS_LIMIT,BUY,19.43000000,0.00000000,2025-03-29 22:38:17.693,2025-03-29 22:47:48.977,0.00000000
383583165,BANANAUSDT,web_489e81d4a99047b5be951b956b1a4da2,19.42000000,101.26000000,101.26000000,1966.00041000,FILLED,GTC,STOP_LOSS_LIMIT,BUY,19.41000000,0.00000000,2025-03-29 22:49:37.490,2025-03-30 02:34:30.003,0.00000000
385499582,BANANAUSDT,web_c8869a48bab845a68f0b996ab9c38919,24.00000000,101.15800000,0.00000000,0.00000000,CANCELED,GTC,TAKE_PROFIT_LIMIT,SELL,24.10000000,0.00000000,2025-03-31 17:18:08.710,2025-04-01 05:18:06.711,0.00000000
386446134,BANANAUSDT,web_ae0fa6833e7542f59235a27afae844df,0.00000000,101.15800000,101.15800000,2104.11140000,FILLED,GTC,MARKET,SELL,0.00000000,0.00000000,2025-04-01 05:18:10.502,2025-04-01 05:18:10.502,0.00000000
386831406,BANANAUSDT,web_af5aa18f2fdd45c9bd581b7f5b5c9be3,0.00000000,104.84600000,104.84600000,2102.46324000,FILLED,GTC,MARKET,BUY,0.00000000,0.00000000,2025-04-01 10:20:40.155,2025-04-01 10:20:40.155,2102.48038000
387415847,BANANAUSDT,web_14a74d5fa4a24a1494b967ce9fbf7f0e,24.00000000,104.74100000,0.00000000,0.00000000,CANCELED,GTC,TAKE_PROFIT_LIMIT,SELL,24.05000000,0.00000000,2025-04-01 16:21:50.289,2025-04-04 16:28:39.915,0.00000000
392834280,BANANAUSDT,web_2824c52c592b4098944bb8ae0137f50c,20.20000000,104.74100000,0.00000000,0.00000000,NEW,GTC,TAKE_PROFIT_LIMIT,SELL,20.21000000,0.00000000,2025-04-04 16:29:49.769,2025-04-04 16:29:49.769,0.00000000


In [42]:
# sql = df_to_sql_insert(ordersdf, 'orders')
# sql
ll = []
# for idx, row in zip(ordersdf.index, ordersdf.to_numpy()):
for row in ordersdf.itertuples(index=True):
    dd = row._asdict()
    dd['id'] = dd['Index']
    del dd['Index']
    order = Order(**dd)
    print(order)
    # dict_ = {
    #     'id': idx,
    #     'symbol': row[0],
    #     'price': row[3],
    #     'amount': row[4],
    #     'executed_amount': row[5],
    #     'cumulative_amount': row[6],
    #     'status': row[7],
    #     'time_in_force': row[8],
    #     'type': row[9],
    #     'side': row[10],
    #     'stop_price': row[11],
    #     'iceberg': row[12],
    #     'implemented_at': row[13],
    #     'quote_amount': row[15],
    #     # 'exchange_id': 1,
    # }
    # print(dict_)

InvalidRequestError: When initializing mapper Mapper[Taxonomy(app_taxonomy)], expression 'Account' failed to locate a name ('Account'). If this is a class name, consider adding this relationship() to the <class 'models.common_models.Taxonomy'> class after both dependent classes have been defined.

## Trades

In [2]:


# async def trade_history():
#     # bsm = BinanceSocketManager(client)

account_trades = await client.get_account()
# traded_symbols = {balance['asset'] + "USDT" for balance in account_trades['balances']}  # Adjust for different pairs
# traded_symbols = {bal['asset']: bal for bal in account_trades['balances'] if float(bal['free'])}
# traded_symbols = [bal for bal in account_trades['balances'] if float(bal['free'])]

# Fetch trades for each symbol
# tasks = [client.get_my_trades(symbol=symbol) for symbol in traded_symbols]
# all_trades = await asyncio.gather(*tasks, return_exceptions=True)
# ic(all_trades[0])

trade_history = [bal for bal in account_trades['balances'] if float(bal['free']) > 0 or float(bal['locked']) > 0]

df = pd.DataFrame(trade_history)  # noqa
df

NameError: name 'client' is not defined

In [190]:
df = df[(df['free'].astype(float) >= 1) | (df['locked'].astype(float) >= 1)]
# df['asset'].unique()

In [234]:
tasks = [client.get_my_trades(symbol=f'{symbol}USDT') for symbol in df['asset']]
trades = await asyncio.gather(*tasks, return_exceptions=True)
tradesdf = pd.concat([pd.DataFrame(trade) for trade in trades], ignore_index=True)
tradesdf['time'] = pd.to_datetime(tradesdf['time'], unit='ms')
tradesdf = tradesdf.set_index('id').sort_values(by='time').sort_values(by='id')
tradesdf

,symbol,orderId,orderListId,price,qty,quoteQty,commission,commissionAsset,time,isBuyer,isMaker,isBestMatch
id,,,,,,,,,,,,
2710807,REDUSDT,36848173,-1,0.53980000,172.10000000,92.89958000,0.17210000,RED,2025-03-14 16:21:05.713,True,False,True
2710808,REDUSDT,36848173,-1,0.53980000,66.30000000,35.78874000,0.06630000,RED,2025-03-14 16:21:05.713,True,False,True
2710809,REDUSDT,36848173,-1,0.53980000,13.90000000,7.50322000,0.01390000,RED,2025-03-14 16:21:05.713,True,False,True
2710810,REDUSDT,36848173,-1,0.54010000,4.20000000,2.26842000,0.00420000,RED,2025-03-14 16:21:05.713,True,False,True
2710811,REDUSDT,36848173,-1,0.54010000,21.30000000,11.50413000,0.02130000,RED,2025-03-14 16:21:05.713,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
39390706,ZROUSDT,845588788,-1,3.22300000,103.50000000,333.58050000,0.10350000,ZRO,2025-03-27 06:59:30.675,True,False,True
39390707,ZROUSDT,845588788,-1,3.22300000,99.81000000,321.68763000,0.09981000,ZRO,2025-03-27 06:59:30.675,True,False,True
118591245,EGLDUSDT,1910149777,-1,18.85000000,9.22000000,173.79700000,0.00922000,EGLD,2025-03-27 12:32:15.535,True,False,True


In [237]:
# tradesdf['symbol'].unique()
tradesdf.columns

Index(['symbol', 'orderId', 'orderListId', 'price', 'qty', 'quoteQty',
       'commission', 'commissionAsset', 'time', 'isBuyer', 'isMaker',
       'isBestMatch'],
      dtype='object')

In [ ]:
tradesdf[tradesdf['isBuyer']].sample(10)
tradesdf[(tradesdf['isBuyer']) & (tradesdf['symbol'] == 'BANANAUSDT') & (tradesdf['orderId'] == 383583165)]

In [271]:
comm = 0.001
total = 19.41000000 * 11.87900000
# fee = total * comm
expense = total + 0.01187900
# fee
# total
expense

230.58326899999997

In [238]:
tradesdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 116 entries, 2710807 to 118591247
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   symbol           116 non-null    object        
 1   orderId          116 non-null    int64         
 2   orderListId      116 non-null    int64         
 3   price            116 non-null    object        
 4   qty              116 non-null    object        
 5   quoteQty         116 non-null    object        
 6   commission       116 non-null    object        
 7   commissionAsset  116 non-null    object        
 8   time             116 non-null    datetime64[ns]
 9   isBuyer          116 non-null    bool          
 10  isMaker          116 non-null    bool          
 11  isBestMatch      116 non-null    bool          
dtypes: bool(3), datetime64[ns](1), int64(2), object(6)
memory usage: 9.4+ KB
